# Suspension

Springs, anti-roll bars and tire stiffness give each axle's ride and roll rates (`lapsim/subsystems/suspension.py`);
the kinematics come from the OptimumK hard-point exports (`cars/ct17_ic/hardpoints/`, solved in 3-D by
`lapsim/subsystems/hardpoints.py`): camber, toe and roll-centre height over wheel travel, anti-dive / squat,
motion ratios and the ARB blade rate. The design report is the table design judges ask for.

In [ ]:
%matplotlib inline
from nb import *

In [ ]:
CAR = lapsim.load_car("ct17_ic")
SL = [slider(CAR, "vehicle:front.spring_rate", 100, 500, 25, "front spring", 1 / 175.127, "lbf/in"),
      slider(CAR, "vehicle:rear.spring_rate", 100, 500, 25, "rear spring", 1 / 175.127, "lbf/in"),
      slider(CAR, "vehicle:front.arb_blade_length", 1.5, 5.0, 0.05, "front ARB blade", 1 / 0.0254, "in"),
      slider(CAR, "vehicle:rear.arb_blade_length", 1.5, 5.0, 0.05, "rear ARB blade", 1 / 0.0254, "in"),
      slider(CAR, "vehicle:mass.cg_height", 200, 350, 1, "CG height", 1e3, "mm"),
      slider(CAR, "vehicle:mass.front_weight_fraction", 0.40, 0.60, 0.005, "front weight fraction")]
out, out_lap = W.Output(), W.Output()


def draw(*_):
    with out:
        out.clear_output(wait=True)
        veh = with_values(CAR, SL).build()
        veh.pt.unlimited = True           # tire-limited balance, as the website shows it
        axle, carwide = lapsim.subsystems.suspension.design_report(veh)
        veh.pt.unlimited = False
        fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
        for A, nm in ((veh.Af, "front"), (veh.Ar, "rear")):
            k = A.k
            ax[0].plot(1e3 * k.travel, k.camber, label=nm); ax[1].plot(1e3 * k.travel, 1e3 * k.rc_height, label=nm)
            ax[2].plot(1e3 * k.travel, k.toe, label=nm)
        ax[0].set(xlabel="wheel travel [mm, + bump]", ylabel="deg", title="Camber change"); ax[0].legend()
        ax[1].set(xlabel="wheel travel [mm]", ylabel="mm", title="Roll-centre height")
        ax[2].set(xlabel="wheel travel [mm]", ylabel="deg", title="Toe (bump steer)")
        plt.show()
        display(axle.round(3)); display(carwide.to_frame("value"))


def laps():
    display(with_values(CAR, SL).run(("skidpad", "autocross")).table().round(3))


for s in SL:
    s.observe(draw, "value")
draw()
display(W.VBox([*SL, out, run_button("Run skidpad + autocross", laps, out_lap), out_lap]))